# Run-Mode Screening Workflow

Learn how to structure a two-stage screening campaign with a Python oracle (run mode).

**Scenario:**
- Large compound library (30K real compounds, ID + SMILES only)
- A cheap first-stage oracle and a more selective second-stage oracle (pharmacophore matching as a proxy for an assay)
- Goal: Find top performers with minimal measurements
- Budget: stage 1 measures 1% of the library per cycle, stage 2 measures 1% of the pruned pool per cycle (the notebook prints the totals)

**What you'll learn:**
- Hierarchical screening (fast → slow oracles)
- Narrowing the library between stages using the model's predictions
- Reusing the feature cache between runs
- Reading per-cycle predictions from the `prediction_cycle_N.parquet` files
- Result export and analysis

**Key Note:**
- **Run Mode**: Oracles generate scores on-the-fly (no ground truth)
- **Metrics**: Selection quality metrics (avg scores, not enrichment)
- **Focus**: Workflow structure and measurement efficiency

**Time estimate:** 20 minutes

## Section 1: Problem Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../..'))

from learnm8 import run_active_learning
from examples.oracles import SimilarityOracle
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load large library (30K real compounds, ID + SMILES only)
# A real campaign could use a far larger library
large_library = pl.read_csv('../data/ampc_30k_no_scores.csv')

print(f"Library size: {len(large_library):,} compounds")
print(f"Columns: {large_library.columns}")
print(f"\nGoal: Find top 100 binders")
print(f"Strategy: Hierarchical screening (fast → slow)")

# Setup persistent cache
cache_dir = Path('.cache')
cache_dir.mkdir(exist_ok=True)

## Section 2: Stage 1 - Fast Filtering

In [ ]:
# Stage 1: Fast 2D similarity screening (the whole library is scored by the model, a subset is measured)
print("\n=== Stage 1: Fast Similarity Filter ===")

similarity_oracle = SimilarityOracle(
    reference_smiles='c1ccc(cc1)C(=O)O',  # Benzoic acid scaffold
    fingerprint_type='morgan',
    metric='tanimoto'
)

stage1_results = run_active_learning(
    compound_pool=large_library,
    oracle=similarity_oracle,
    target_col='similarity',
    learner='rf',
    featurizer='morgan',
    n_cycles=10,
    batch_fraction=0.01,  # 1% per cycle
    cache_dir=cache_dir,
    score_direction='higher',  # Higher similarity is better
    output_dir='results/stage1_results',
)

print(f"✓ Stage 1 complete")
print(f"  Measurements: {len(stage1_results['compounds_df'].filter(pl.col('status') == 'labeled'))}")

## Section 3: Stage 2 - ML-Guided Selection

In [ ]:
# Narrow the library for Stage 2 using the Stage 1 model
print("\n=== Stage 2: ML-Guided Selection ===")

# Per-cycle predictions are written to prediction_cycle_N.parquet files (columns: ID, prediction),
# not stored on compounds_df. The last file holds the final Stage 1 model's predictions for the
# compounds that were still unlabeled at that point. Keep the top 10% of them by predicted similarity.
last_prediction_file = stage1_results['prediction_files'][-1]
top_10_percent = int(len(large_library) * 0.1)
top_predicted = (
    pl.read_parquet(last_prediction_file)
    .sort('prediction', descending=True)
    .head(top_10_percent)
)

pruned_pool = large_library.join(top_predicted.select(['ID']), on='ID', how='inner')

print(f"Pruned library: {len(large_library):,} → {len(pruned_pool):,} compounds")

# Stage 2: Intensive ML-guided screening with a more selective oracle
# In a real campaign this would be an expensive oracle (docking, assay, etc.)
# Here we simulate with a more complex pharmacophore oracle
from examples.oracles import Pharmacophore2DOracle
from learnm8 import CycleConfig

expensive_oracle = Pharmacophore2DOracle(
    reference_smiles='c1ccc(cc1)C(=O)N[C@H](C)C(=O)O',  # More complex target
    metric='tanimoto'
)

stage2_results = run_active_learning(
    compound_pool=pruned_pool,
    oracle=expensive_oracle,
    target_col='pharmacophore_score',
    learner='rf',
    featurizer='morgan',
    cycles=[
        CycleConfig('random', n_cycles=1, batch_fraction=0.01),
        CycleConfig('ucb', n_cycles=3, batch_fraction=0.01, acquisition_params={'beta': 10.0}),  # Exploration
        CycleConfig('greedy', n_cycles=6, batch_fraction=0.01),                                  # Exploitation
    ],
    cache_dir=cache_dir,
    score_direction='higher',
    output_format='csv',
    output_dir='results/stage2_results',
)

print(f"✓ Stage 2 complete")
print(f"  Measurements: {len(stage2_results['compounds_df'].filter(pl.col('status') == 'labeled'))}")
print(f"  Top score: {stage2_results['compounds_df'].filter(pl.col('status') == 'labeled')['pharmacophore_score'].max():.3f}")

## Section 4: Final Analysis (Run Mode Metrics)

In [ ]:
# Identify top candidates based on the final Stage 2 predictions
print("\n=== Final Results ===")

# The last prediction file belongs to the final cycle; it covers compounds that were not yet measured
final_predictions = pl.read_parquet(stage2_results['prediction_files'][-1])
top_100_candidates = (
    final_predictions.sort('prediction', descending=True)
    .head(100)
    .join(pruned_pool, on='ID', how='left')
)

# Calculate measurement efficiency
total_measurements = (
    len(stage1_results['compounds_df'].filter(pl.col('status') == 'labeled')) +
    len(stage2_results['compounds_df'].filter(pl.col('status') == 'labeled'))
)

print(f"Total measurements: {total_measurements}")
print(f"Identified top 100 predicted candidates")
print(f"Measurements vs library size: {len(large_library) / total_measurements:.1f}x fewer measurements than measuring every compound")

# RUN MODE: Report selection quality (not enrichment)
stage2_measured = stage2_results['compounds_df'].filter(pl.col('status') == 'labeled')
print(f"\nStage 2 Selection Quality:")
print(f"  Average score (selected): {stage2_measured['pharmacophore_score'].mean():.3f}")
print(f"  Top score found: {stage2_measured['pharmacophore_score'].max():.3f}")
print(f"  Score std dev: {stage2_measured['pharmacophore_score'].std():.3f}")

# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Stage comparison
stage_data = pd.DataFrame({
    'Stage': ['Stage 1', 'Stage 2'],
    'Measurements': [
        len(stage1_results['compounds_df'].filter(pl.col('status') == 'labeled')),
        len(stage2_results['compounds_df'].filter(pl.col('status') == 'labeled'))
    ],
    'Compounds': [len(large_library), len(pruned_pool)]
})

axes[0].bar(stage_data['Stage'], stage_data['Measurements'])
axes[0].set_ylabel('Measurements')
axes[0].set_title('Measurements per Stage')
axes[0].grid(True, alpha=0.3)

# Selection quality over time (RUN MODE)
stage2_metrics = pd.DataFrame(stage2_results['cycle_metrics'])
axes[1].plot(stage2_metrics['cycle'], stage2_metrics['avg_score_selected'],
             marker='o', linewidth=2, color='green')
axes[1].set_xlabel('Cycle')
axes[1].set_ylabel('Avg Score (Selected)')
axes[1].set_title('Stage 2 Selection Quality')
axes[1].grid(True, alpha=0.3)

# Best score discovery progress
stage2_measured = stage2_results['compounds_df'].filter(pl.col('status') == 'labeled')
stage2_best_scores = []
for cycle_num in range(len(stage2_metrics)):
    cycle_measured = stage2_measured.filter(pl.col('labeled_cycle') <= cycle_num)
    stage2_best_scores.append(cycle_measured['pharmacophore_score'].max())

axes[2].plot(stage2_metrics['cycle'], stage2_best_scores,
             marker='s', linewidth=2, color='purple')
axes[2].set_xlabel('Cycle')
axes[2].set_ylabel('Best Pharmacophore Score')
axes[2].set_title('Best Score Discovery Progress')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 5: Practical Considerations

In [ ]:
# Export results for review
print("\n=== Export Results ===")

# Export top candidates (ID, predicted score, SMILES)
top_100_candidates.write_csv('top_100_candidates.csv')
print("✓ Saved: top_100_candidates.csv")

# Export the final compound table (status, selected/labeled cycle, measured score)
stage2_results['compounds_df'].write_csv('stage2_compounds.csv')
print("✓ Saved: stage2_compounds.csv")

# Cache statistics
print(f"\n=== Cache Statistics ===")
print(f"Cache directory: {cache_dir}")
cache_files = list(cache_dir.glob('*.h5'))
total_size = sum(f.stat().st_size for f in cache_files) / (1024**2)
print(f"Cached features: {len(cache_files)} files ({total_size:.1f} MB)")
print("💡 Features are cached on disk, so later runs on the same compounds skip featurization")

# Saved outputs
print(f"\n=== Saved Outputs ===")
print(f"Stage 2 results directory: {stage2_results['output_dir']}")
for name, path in stage2_results['saved_files'].items():
    print(f"  - {name}: {path}")
print(f"  - {len(stage2_results['prediction_files'])} per-cycle prediction files (prediction_cycle_N.parquet)")

## Workflow Summary

**Hierarchical Screening Strategy:**
1. **Fast Filter (Stage 1):** 2D similarity oracle; the model's predictions keep the top 10% of the library (30K → about 3K compounds)
2. **ML-Guided (Stage 2):** Active learning on the pruned library with a more selective oracle
3. **Validation (Stage 3):** Expensive measurements on final candidates (not shown)

**Key Techniques:**
- ✓ **Caching:** HDF5 feature cache, reused by both stages
- ✓ **Narrowing the library:** Use the model's predictions to reduce the search space between stages
- ✓ **Saved outputs:** Every run writes its results, metrics and per-cycle predictions to `output_dir`
- ✓ **Efficiency:** The notebook prints the total number of measurements compared with the library size

**Run Mode Metrics:**
- ✅ Average score of selected compounds
- ✅ Top score found per stage
- ✅ Measurement efficiency (reduction factor)
- ✅ Selection quality trends
- ❌ Enrichment factor (no ground truth)
- ❌ Top-K discovery (no ground truth)

**Checklist:**
- [ ] Set up a persistent cache directory (`cache_dir='.cache'`)
- [ ] Configure an output directory per run (`output_dir='results'`)
- [ ] Enable CSV export for all stages (`output_format='csv'`)
- [ ] Monitor selection quality metrics (avg score, top score)
- [ ] Validate top predicted candidates with the expensive oracle
- [ ] Document experimental parameters

**Practical Considerations:**
- Start with conservative batch sizes (1-2% per cycle)
- Use a learner with uncertainty (`rf`, or an ensemble) if you want uncertainty-based strategies
- Balance exploration/exploitation based on oracle cost
- Hierarchical approach: fast filter → ML-guided → expensive validation
- Reuse the feature cache across runs on the same compounds
- For assays that take days or weeks, see the Lab Campaigns tutorial in the documentation (`learnm8 suggest`)

**Benchmark vs Run Mode:**
- **Benchmark Mode** (Notebook 1): Retrospective analysis, enrichment metrics
- **Run Mode** (Notebooks 2-4): Prospective screening, selection quality metrics
- For a real screening campaign, use run mode